# Linear attention, Part 1 of 3: feature maps

*This three-part problem is adapted from 2026 Round 2 Problem 1 (new setting, numbers and wording). Part 2 is u15-p06 and Part 3 is u15-p07; each part's functions are reused by the next.*

A team is building an attention model for vibration signals of up to 4096 time steps, where the $T \times T$ score matrix of softmax attention is too large. Linear attention replaces $\exp(q^\top k/\sqrt d)$ by $\phi(q)^\top \phi(k)$ for a feature map $\phi$. In this part you write two feature maps. Both act on the last axis of a tensor of shape `(..., d)`.

**1. The 1 + ELU map** (feature size $\ell = d$), entry by entry:

$$\phi_{\mathrm{elu}}(x) = \begin{cases} x + 1 & x \ge 0 \\ e^{x} & x < 0. \end{cases}$$

```python
def elu_feature_map(x):
    # Your code: same shape as x
```

The output must be finite, and so must its gradient, for inputs as large as $\pm 1000$.

**2. The second-order Taylor map** (feature size $\ell = 1 + d + d^2$). For $x \in \mathbb{R}^d$ return the concatenation

$$\phi_{\mathrm{T}}(x) = \Big[\ 1,\ \ \frac{x_0}{d^{1/4}}, \dots, \frac{x_{d-1}}{d^{1/4}},\ \ \frac{x_a x_b}{\sqrt{2d}} \text{ for } a = 0, \dots, d-1 \text{ and } b = 0, \dots, d-1 \text{ (row by row: } a \text{ outer, } b \text{ inner)}\ \Big],$$

so that $\phi_{\mathrm{T}}(q)^\top \phi_{\mathrm{T}}(k) = 1 + s + s^2/2$ with $s = q^\top k / \sqrt d$, the first three terms of the Taylor series of $e^s$.

```python
def taylor_feature_map(x):
    # Your code: shape (..., 1 + d + d * d)
```

Both functions keep every leading axis, return the dtype of `x`, let gradients flow to `x`, and must not modify `x`.

**Example.**

```python
elu_feature_map(torch.tensor([-1.0, 0.0, 2.0]))     # tensor([0.3679, 1.0000, 3.0000])
taylor_feature_map(torch.tensor([1.0, 2.0]))
# tensor([1.0000, 0.8409, 1.6818, 0.5000, 1.0000, 1.0000, 2.0000])
#         1       x / 2**0.25       x_a x_b / 2 for (a, b) = (0,0), (0,1), (1,0), (1,1)
```

**Restrictions.** Only `import torch` and `import math`. You may not use `elu`, `ELU`, `celu`, `selu` (functional or module forms): write the formula.

**Constraints.** Up to $10^6$ entries in `x`, $d \le 32$ for the Taylor map. Time limit 1 second per call.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem5_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem5_submission.py
import math
import torch


def elu_feature_map(x):
    # Your code
    return x


def taylor_feature_map(x):
    # Your code
    return x


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem5_submission as submission
importlib.reload(submission)
# e.g. submission.taylor_feature_map(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9Wwt32zaW/itY9ZwJ6cqMrDymo9TZdVK3kzl5nTg9M3tsLQuJkMwxRTIEFUv1+L/vdy8AEpTkVzq76aklgsDFfb8AXfXSpDcSveXBs/1y8KzXF72FqiWGrrbf1GmdKRp7m+ZKVkLWtcrrtMj74qOsanEgipl4MhIzJetlpcRClprWZUrrInfAssFz3ibNl7XSGD14hscknc3S6TKr1xh5ggFAruh7r5brrKhiCzQGUF5eJMtMxblcMEZlVUwytXgW6+VkkWoNpGiWzLLiUiVxuiiLqqbNTnt1UU3PGYSsz3tjfJnIPFcJv1TZkl4dv/2VPqb28bV91vb5hJ7tyjgripIgz2SmFXFJ6Tqu04UqlnVML4YDGpVzs33DtChjLjKazdgGlXJZF/NKJlEuc95QF8tqyhTLRJa1SvaHg+Hz/U/D/Y8HtKCEHIZ4XVdLdY3nWZoxj69657LKIYeoZKae8X9/VVmpKi3qc1kLy0JB+9Gg4ZkIfptVxULY5W5077cwOsvP8iM7XaRaTIucyHi8zNNaP/6RPl4+tlDxXKbJy8dmNpCIxJtaJGoGHmB/sEzMlvmUeKAFCTXhwXjP4FbLC4Vv6ixv5SuMCojfMPRbJI7/cfTu49vjE5GluiaSlAFLwLTQ58VlLrDo0zIXwddUp8AqfCFOCFwtqiW2VV9VteZFIO1nmWYQhBaVTLUS9Cgu0/pcSAF4Ws6VuKxSkpuYFZV4pfKR0HItLgndS6nxtsjnfZETVJEsF6U4T5ME09O8XNaauGekkFuWToty3TyQ/hB/7WOO9WsBqHlJo9+J10UmJ+L1x1+1kLA0TWpeEa9yQuWRFu+G4mNV9IUuRFZMZUYQ03yuxWIJpkwUMRUsKsjqUrzOUuLDBPSfQ5QzOYWZRLTTMBqQcKWYL0n+gdlZA1am9uvzSslE6FJBYJhVFcv5ebaGtmQzAVyAxbSoiNGVIqbXIp0Rgnhkg7R0WGshppx8PD7+Kf756PXnD5/EIe1OBJ/l00xidxJDcLyaqpI0JRyd5QL/Srwys6BQ4LhKAuhi0hcLPXdzsG9eEJPzxI7Qv1a4Ac1tgcR5GazcWjgib42TELkRb3QG+tNc1zKfqmDVN++jzyrXRRV6y3lbBSPPxSpK4Gun50EYTcsl/rKYg9BMVkymeMPbHVdVUXlQDMkerLyMpJZVJdfAu6VDn8tSBfOi7kMpc/oL/Tw860Gz4WvPeg4zDV7XyzJTAVGO+WHESy0uzFPMcZMIVtgXs7PeFQG8hsC12UtcafgdtSrVFN5JXHnzr7Fdi9k0K/RtmPVFVRfZ4YHaf95HpDFff3D4wrIugXKDrNT1GoTOskISYjTOe3ZfeMTMDX1E0qX5egM5hgA7PdxFm3vF5DUbQCCphntLa6wGHlkWhAKegrTQf3dp3/nbkyeVKXvCnNak+awBbhWZ5J1llolghmUX/bHsoj++4qmqAseYEYTcQq7oQ050MBf74jIMQ4INQtPflVCIZWJAxrfDVLp8YkcnvsoM/kEEmazm5HYVKay4wscoejK/DjuyV1+WMruHVnaNykwlpPOENb0fMkeBBAJoZd+f9VgcLZAd6uIpSDvn/1wtfNVg/GPLByC3ZUw+VxsIJJXRJsZQBsKWiW9hwBlf4dV/VB3UaBaGOsJIdcwmwuKgb/pGefCGnkTaJVv40yhYYQGHUcypWhx30JnD/ZHk7K6P3KRHDmoX01jnyCjPi/r/1zVnRa4e6JSxpdPKFakkx/ZeiJCYsPV28QmSdApGUuIS+vh0ietgRwCDRKny8DMyPU/BLIKfwb5N9LYBhJ0QQkMRweR3qw7jIZZA9sXk2/ku78V3M8XYBe/3rWyXxHaGo7uuwG4kzSY6mIS7fA2Wt35m5/LGT0zsV5Zt17KJgr7xdtDs3JeVg8MQvOSliBdLGAWym2CGumoP7hSWsbd3cUnfHCYmdXyNCCBmOe88owSVaECOgyxwDvOSOVLGar5cIClHCBFlJqcqclknR32yJ/jF09ayZMj5rKQFtOPYzER2RsEjDzYRMq9pSQpFBsE6pKUKqYyqJOLb72kZmDW8R0fB2Z80ymWdyBolDnQxUZTgp7OUMktKmVtavrtKxffi4PqFuCyqC8rqJSuvIPEhH238pWUykG85TEl8mgdaUSoIrO7L5Rc+i+Fd/ZR7RybNlSfioUbuDQqIQZzrPmpWInk3NRRt9QEgKirBUCAtNJHEefg7ORVTbEBLQKTJuqla4cSekgNUF/uTZQJPSsOcgyMtT6mAWpYwJ1FfplNFAMGhFBG6mFkLBs6Oo5pViEZmyJcofBMErFjiXSQ+04u0wjCIyR7RFhQXzGbEYZV01KoeUDYJCBGqy1nMYFTlvM0dqpTUuxcjR6ltOmLpPRRWiOKx8MuGViVjUsWKrCEY+noHIWKbHw8tpA1XNEFNc9EOyWFfXAyxW9c/Msr9jcEOJXezgvEkRgyZDUNvmNmAii1IKMTewA4/3/QJYjuqi4L1BsV/Uo+i4ewaWT60TL2wVVejxKxMMDsGJJY51fNXljV2HdeQKOyciHf84y1RWaIMnkG15pTAmiLzysqJAZF13GqgVT6HdarECawtcCDJpFiA2zMJ9YubiWfc61lqKdMi5g5Dp8vxC43AAoBZ2zxI2Wmcp1lCVjelzggK5pRsgWb+7eTDe0ENGrIcXSfAkG21XNfnGNl/I9w+4kfb4oiTtHopfmz3wEs8wyr/xWP1y62SH8bmlf8Q7KrO0kkzZD4wEi3rNGuHi+brP3XRNglKWZ/7y3U6z2W7TK/1RnvBPVSICxM5vWhGoMU5NQtMr0FPU5CZq8bdYeAirfczhWniIHoeHdicTo+oT7Sgfl+fXQqqCxt8nPMrZQouz89yt0c0SzNotHsMznpAu6gUVYC204JMNNr7Nb/IqYeji4w6KgV3ALBhWpLUoj2bKf71+NMxLMfyIvqIzyCOqQ0Wx2EEx0XLkdWVsgJmZzm4EtHkCNqqqjoYwBiQPhCYEAB39b64FyS+Q6z+Ikfi+OlgyJ2wt38/+u+T+NXR+/fHPwGDq7NebPuOcUzEgDAnT/OoVmpqvyHPN9+mxaIErme967P87YcPH+P3H346PgG4ALoS/VzAVunL36G4ynw90ut82rx4ixzyNWCYpxPlPfyEJLN9+kXlFJ+L6nhVhn6L5bPpxdzVZYklKr1FQDq2XMDfIGypxmK5VHSA/G4EZTbTeHquphc60NWURFzLbrz9ZOydsmHqJ4OPxYVhzkLPwZgXcIuUAZoeH5uNrh810uFkyLYSKVDVVWraip0QZYLQ6XhXNltXCokdcwlagvoaiHbz0JM16vMV56HUlFPb+eGpRXsEbeG2sMV+RG5S83JbIhe58TNXihvCeXE9ou+YjeLn2iJo+9gUEsCvCI4ZdrLR3CYenY4tohPZjHOcrF3I4XZovNQMzBufwB5i0yu146TiSBlhu2lC3WlkYpVJFAxgQDDzA6poKOlOJ9TRBzkXnLupLCLFthtwm9zj+ULWi2Xmj8y4OwJXTb4ZzL+U2UVAwtgI3V6aTtONPpvSYLOqaFJZmhgxuqPtAGZbKsiXA8mTKClfUGvBPEbYDYykzDFYIPOEO6JijoAvGFkjiHC0OzZ6sohkktg9wu3JrWy8eZEus5TETXue7h+MdyxshRcty4Sy7h1r/SIxu52NP8PrbVIDgUJWzEfbeAf5rUHt4CWvsIzE9z/AxU0OAtoGFzzWdTiwSwM2lt7EvG9aynQ+lOlHznY2ad9QCMaDDO1eUN97Hvk2gGlyM1W7J+3asQ1Wm3uy4UeyLFWeBK7jYxad9YzLM+79P7v82rULR8FXaf7BRrGj5fxIUwwKTW9lY0FU2HnvZP0OKeOWThsf9EDkgJinkB5IShItqBvdv8sCjJUYladRNgiaF/2zoPKUHWzg7ROGu73L9xRRwBgH6upRXzwyQJxFkQE+wn6oe+ePqBV7vbufuAt/alnsQF873Ftg3zUniBOZa+Lt9NxGCE4F2wDB7WclUcT+gz/DF6YazS7lWu+b01hUMsgL0wYQKaWFVuTZuu/2ROXShprAaIFpItlUipKOtcjUHKX5JaVNgmdGURR6wQxJmUgKEooBfJ5yfWmkEHhB808cHL0obNC1wRdq+C8ReLH0T6KTF3oqhB0erDrAQAvLH4If6HCH7gBy+EcknBduj+ZItpUyMO8SHxsjiovS5JPdbhtNN0a2YXr3IZgqSujFfwkHm0pWz25EYDInu8PpYOwr9w7yH8KCdtfbiGfntkUyj34bwSZNYmXvEIphMmNS6kqdq5yL2CZ35FX/XvotZde3VOqzKgWYbB1zJhsABArdmKqpbkb/IVf7jGei9LRKuaoQJVQV9j0X9iSaJ6Tu+JoKNpironmaD+u7HaYJtVVc3Rqhbsb3Oq4ngYriuHmBks8mmwQcmeaMg/qM9gEISGpG5adyOR9Vfg0JNiu9pJYJNazIy3Jv0iod/lI65mXtnSESBSsMzebjtBZ9y0w6MeETEuWdj1D2f33Fu17TfMfsBTLxoKnF4Gk3q1yqZGU1/3rapIiOlBtnDsdeSWw5RQn4oWimPHHpOvQe49RwiLJCJjoIGInHpDZkE9yK6BE8xCzqZATO2aF8IogWGX+C4wZ3Ew/Ftl2YgpGGTsf0TKWebh9NY8aorK2kSRNp4H0BEfjwT1toY0LnhlK0MXTKYMkE9anBaszRgp7YSsZ+OutKOpfPau47bm3rewnuWgTMTrocogMz3Y/yRktcy3MGpNMiOqmpffLmg2OeafJE5iOwTydvfjl6++ld31bpdircWczNXXcK7Pmy7qUl4uVw4HDpVsd8CaZtVkGcSVopGJ8RRgBMN9Msh6OqU9qicmi++fzm3fGn+NPx0dt+g91GmqNLRdrTbYVFNBpTb8Z0dqg7yUcmHkXeBTGjGm1jDmLyXNXGfsvJ9nYWFm9IWwf0Z3MhLMbM06f0mguGsVH8bZLYhlQVUQ8oNsvI84TfwLhBtzdh2y6dloRVRGscY+PNDI3kgdmrsceti+KCzxTElZPHKBpQB9i5LrvLK6lV0yEyPZC2Jfbq7fFgcNAi8CAybsLYHgwxmnQSQzfECisomwTdEo8as94CvYtRzq/QvtBn0ig+iQ9QgD8dDAaj8TcbMl9r7PZxeMi/32HmuLNid6IJWvrmVXi3cFteJYUySZO5bIfoQiCu/br9gSTMH2aTZz1z6Y+ssAkZzT3Ahm57j/Bu25t7xje/yZYMNDvLXvezjS1X5Zkp5Bnc1UGXxFu3R7GGkoYAhcrMOHb6Rr79q6y02yNKa7XQAd+eybtRgb2qO/enAxQJNIJZ6De84CXoEJJTEtpxI3nmcEzaXy3ht4xSULbCtwhyR9qGwyX3nOZL5ctvagKs8YcjBnL6bDSG/+ZzYWBru9SC3eNZz4J2k73d6C11g6ns2VWmWH9rDn1oaBANrh9yJrZ98eEbos4fjTz2cG7bK3+TS7NCOG05R5ZKiff23Y0tD94uNrmHsXF21lmRz925zKbPFkGam1tmnM33qYCYyUq4k8HQdwJ2cz7c2Ohp/0GCG5wpvd5xV+VBseTfhYt/z4DPK5J7xpAWVqPgBK+CFifBTQe1ffFkV3iz6ezYFWRV49geHIqaBSZXPaTSMWhz19uyUnYqk6LIgi3EzDsCVXc0l8DVHXDNCleg3BhS+FrzTLiCx3i3OKb6hg7KRu54gModc7LbBAs607XtM7q8355gmotHNx3W9flgu2+unPaFvdbnXbXp21shhFpzjxye33pwtZKLkjzh2DulVrNYZcv2JtoKC1YRhEUuvls0m34Ul3TBSrw8FNCHFd1gcVei1KoMzDdUi+liuWhvqf2uqgK1Vkj/urub30XcjkDCY0wwFabuIM+cCp2aHVAo6WDPzRphWp8wS6gyPQRA+gwJ4ccAt7cHZz581oH0PcXI1WkURX0hR5IIG4s9YUcmowmPhFhPQov0l6oOhpiQeFeNzCUNOzLxR8Y7WInMIuC9gWa6ONw/8FjjS4ydt8ces7rmC2gBGBJBEAP6M4wG4871FP8eFoBEELX/CxSww9VffMEai/oieNIPzbkubz4SG4uaVMfc2uU1rRrdvrK9Df2svQ393MJbb5F2cC+itn9Z0xfrHXT9eYOu7XU3kWZ1dH33+hsJ7MoVfDF3YmM4ptgguVPIdFckD5726bdEz8CMH/pi7o68D82U5gg8CKOFzOmSHl0nCQ7CENr55I+qw8rdHz7rbUynvp2PWti9OU4dOrY6co3WAPlGzwaYkbFRcdUs6FyutSuv76F2d6J3q/ox0hw/CLh4idBrrrTvAj6yP7NpfiNmfo5CbT4E86+qc+G3kTnfKo/NL2dY8hQSEKjruywcEZJMYf+J+Wgtnr+ZUTNpTFz5skRWqRm6f1nTKMEOwQerW3jbXCHeyWRZi+8ZvxGBL5f1HVwmIWtEhjCizualrBLn5OlS9xbp7GoprlgeAId26MnmwAE/HzSMcX/GvohXEfHF9eCpwbZTvnkhnHTACzAAyrjaUEIDyt3+32ZOA6Dh0n09hPUtmVxTWXCLaxhCu/vi6f3dwjB8sDPdFSTMvgeIiU8pLj5n5dhezEZocQzvcK+r+8F4EAMvFNKoLDZXP+q1z8l5w8mbWOWS3S99cbHBdsgS1D93+YV5xe3I5x1ZzMP+N66ztTX2nl1Yq91mTfDFtAB3vbrwlX72pfXFG5vOdvG8ccpuoe+TOwDMUYT9zq6tETNlZ8EXBKGLkG0eCQ6yp6fu5zhGC4AaZszaKUarNP+/J+hW7LDVC2EEKsrzFLSLiL9chOKrblf9z/Dx0FeTv7Rq8pcb1OR2P2xEhzhyt9xuV6in4b288w5hrjYjawfMbrtxHtnvnVF9Rva0QqnNv3IU7re5jdRu9tCryuTmGz8p6eISB52fB7TWXYU3gO360lUV3UxS41F9+f7QyvdgsClg/k3nbWIl+Q2fPTdWeV9BPus6UXsZn8PNQ5xos/fwz0/Y+XGCIKZS356e7t51K5vbcDw2R7g/lc/DcAfSLaQtlHdVCr3r6/8FjFCPdQ==")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
